# DAY 3 보충 — [3-9] 사진 한 장 라벨링 · [3-44] 상담 녹음 전사 다시 하기

**AI 적용을 위한 현장 데이터 수집 및 디지털화** · DAY 3 보충 실습 (2026-10-28)

어제 DAY 3 노트북에서 **[3-9]** 와 **[3-44]** 두 칸이 오류로 멈춘 분들이 있었습니다.
이 노트북은 그 두 실습만 따로 떼어, **처음부터 위에서 아래로 실행하면 끝까지 돌도록** 고친 판입니다.

### 어제 무엇이 문제였나

| 칸 | 어제 멈춘 이유 | 이번 판에서 바꾼 것 |
|---|---|---|
| **[3-44] 상담 녹음 전사** | 녹음이 **3분(181초)** 인데, 말한 사람을 구분하는 전사 모델은 **30초가 넘는 음성이면 「나눠서 처리하라」는 설정(`chunking_strategy`)을 꼭 넣어야** 합니다. 그 설정이 빠져 서버가 요청을 거절했습니다. | 설정을 넣고, 돌려받은 「화자 A · B」를 「상담원 · 고객」으로 바꿔 적습니다. 그래도 막히면 **일반 전사 → 화자 나누기** 두 단계로 자동으로 바꿔 진행합니다. |
| **[3-9] 사진 한 장 라벨링** | ① 사진 데이터셋을 못 받아 사진 목록이 비어 있었거나 ② 모델이 JSON 이 아니라 설명 글로 답해 JSON 읽기에서 멈췄습니다. | ① 사진을 못 받으면 같은 모양의 대체 사진을 만들어 씁니다. ② 모델에게 **JSON 으로만 답하게 강제**하고, 그래도 읽지 못하면 받은 글을 그대로 보여 줍니다. |

> **오류가 나도 이번엔 이유가 보입니다.** 서버가 거절하면 그 이유(영어 한두 줄)를 그대로 출력합니다.
> 그 줄을 캡처해 패들릿에 올려 주시면 바로 확인하겠습니다.

---
### 쓰는 법

1. 「파일 → 드라이브에 사본 저장」 을 먼저 합니다.
2. 왼쪽 🔑(보안 비밀)에 `OPENAI_API_KEY` 가 등록돼 있고 「노트북 액세스」 가 켜져 있으면 키를 묻지 않습니다(어제 등록하셨으면 그대로 됩니다).
3. 맨 위 칸부터 ▶ 를 차례로 누릅니다. 건너뛰면 오류가 납니다.

## 0. 준비 — 키 · 실습 데이터

In [ ]:
# API 키 — 🔑(보안 비밀)의 OPENAI_API_KEY 를 먼저 찾고, 없으면 입력창을 띄웁니다.
import os
from getpass import getpass

def _secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

API_KEY = (_secret("OPENAI_API_KEY") or _secret("GEMINI_API_KEY")
           or os.environ.get("OPENAI_API_KEY") or os.environ.get("GEMINI_API_KEY") or "").strip()
if not API_KEY:
    print("🔑 보안 비밀에 등록된 키가 없어 직접 입력받습니다.")
    API_KEY = getpass("API 키 입력 (sk- 로 시작하면 OpenAI, 아니면 Gemini): ").strip()
PROVIDER = "openai" if API_KEY.startswith("sk-") else "gemini"

OPENAI_MODEL = "gpt-4o"                          # 사진을 읽는 모델
OPENAI_STT_DIARIZE = "gpt-4o-transcribe-diarize" # 말한 사람을 구분해 받아쓰는 모델
OPENAI_STT_PLAIN = "gpt-4o-transcribe"           # 위 모델이 막힐 때 쓰는 일반 전사 모델
GEMINI_MODEL = "gemini-2.5-flash"
print("사용할 서비스:", PROVIDER, "· 키 길이", len(API_KEY), "자")

In [ ]:
# 실습 데이터 — 상담 녹음(audio/)과 주조품 사진을 준비합니다.
import os, glob, zipfile, urllib.request, subprocess, sys
RAW = "https://raw.githubusercontent.com/aebonlee/materials/main/build-data/"

if not os.path.exists("audio/call1_engine_start.wav"):
    try:
        print("data_day3_docs.zip 받는 중 …")
        urllib.request.urlretrieve(RAW + "data_day3_docs.zip", "data_day3_docs.zip")
        zipfile.ZipFile("data_day3_docs.zip").extractall(".")
    except Exception as e:
        print("  받지 못했습니다:", type(e).__name__, e)

# 사진: ① 공개 데이터셋(kagglehub) → ② 대체 사진(casting_local)
def_files, ok_files = [], []
try:
    import kagglehub
    path = kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product")
    base = os.path.join(path, "casting_512x512", "casting_512x512")
    def_files = sorted(glob.glob(os.path.join(base, "def_front", "*.jpeg")))
    ok_files = sorted(glob.glob(os.path.join(base, "ok_front", "*.jpeg")))
except Exception as e:
    print("공개 사진 데이터셋을 받지 못했습니다:", type(e).__name__)
if not def_files:
    if not glob.glob("casting_local/*/*.jpeg"):
        print("대체 사진을 만듭니다. 30초쯤 걸립니다 …")
        urllib.request.urlretrieve(RAW + "data/make_day3_data.py", "make_day3_data.py")
        subprocess.run([sys.executable, "make_day3_data.py"], check=True)
    def_files = sorted(glob.glob("casting_local/def*/*.jpeg"))
    ok_files = sorted(glob.glob("casting_local/ok*/*.jpeg"))

print("결함(def)", len(def_files), "장 / 정상(ok)", len(ok_files), "장")
print("상담 녹음:", "있음" if os.path.exists("audio/call1_engine_start.wav") else "없음 — 위 메시지를 확인하세요")
assert def_files, "사진이 한 장도 없습니다. 위 메시지를 캡처해 패들릿에 올려 주세요."

In [ ]:
# 호출 도우미 — 오류가 나면 서버가 돌려준 이유를 그대로 보여 줍니다.
import json, base64, mimetypes, uuid, urllib.request, urllib.error

def _send(req, timeout=300):
    try:
        return urllib.request.urlopen(req, timeout=timeout).read()
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", "replace")[:600]
        raise RuntimeError(f"서버가 요청을 거절했습니다 (HTTP {e.code})\n{detail}") from None

def chat_json(prompt, image_path=None):
    """프롬프트(+사진)를 보내고 JSON 으로만 답을 받습니다."""
    if PROVIDER == "openai":
        content = [{"type": "text", "text": prompt}]
        if image_path:
            b64 = base64.b64encode(open(image_path, "rb").read()).decode()
            mime = mimetypes.guess_type(image_path)[0] or "image/jpeg"
            content.append({"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}})
        body = {"model": OPENAI_MODEL, "temperature": 0,
                "response_format": {"type": "json_object"},       # ← JSON 으로만 답하게 강제
                "messages": [{"role": "user", "content": content}]}
        req = urllib.request.Request("https://api.openai.com/v1/chat/completions",
                                     data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json",
                                              "Authorization": f"Bearer {API_KEY}"})
        text = json.loads(_send(req))["choices"][0]["message"]["content"]
    else:
        parts = [{"text": prompt}]
        if image_path:
            parts.insert(0, {"inline_data": {"mime_type": "image/jpeg",
                                             "data": base64.b64encode(open(image_path, "rb").read()).decode()}})
        body = {"contents": [{"parts": parts}],
                "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}}
        req = urllib.request.Request(
            f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent",
            data=json.dumps(body).encode(),
            headers={"Content-Type": "application/json", "x-goog-api-key": API_KEY})
        text = json.loads(_send(req))["candidates"][0]["content"]["parts"][0]["text"]
    cleaned = text.strip().removeprefix("```json").removesuffix("```").strip("` \n")
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        raise RuntimeError("모델이 JSON 이 아닌 글로 답했습니다. 받은 글:\n" + text[:600]) from None

print("호출 준비 완료 —", PROVIDER)

## [3-9] 사진 한 장 라벨링 (다시 하기)

어제와 같은 프롬프트입니다. 달라진 것은 **「JSON 으로만 답하라」 를 프롬프트뿐 아니라 요청 설정으로도 강제**한다는 점입니다.
프롬프트는 부탁이고, 설정은 약속입니다. 부탁만 하면 모델이 가끔 설명 글을 덧붙이는데, 그러면 다음 단계 코드가 읽지 못해 멈춥니다.
현장에 붙일 파이프라인이라면 **형식은 설정으로 못박고, 그래도 어긋나면 원문을 남기는** 이 방식이 기본입니다.

In [ ]:
PROMPT = """이 사진은 엔진 냉각수 펌프 임펠러 주조품의 입고 검사 사진입니다.
표면 결함(기공, 균열, 미성형, 버, 흠집 등)이 있는지 판정하세요.
{"라벨": "ok|def", "근거": "한 문장"} JSON만 출력하세요. 결함이 있으면 def, 없으면 ok."""

from PIL import Image
import matplotlib.pyplot as plt
f = def_files[0]
plt.figure(figsize=(3, 3)); plt.imshow(Image.open(f)); plt.axis("off"); plt.title("보낸 사진 (결함 쪽 첫 장)"); plt.show()

r = chat_json(PROMPT, f)
print(r)
print("\n라벨:", r.get("라벨"), "· 근거:", r.get("근거"))

실행하면 사진 한 장과 함께 `{'라벨': 'def', '근거': '…'}` 같은 결과가 나옵니다.
이번에 확인할 것은 **정답 여부가 아니라 「약속한 형식으로 돌아오는가」** 입니다. 형식이 맞아야 60장으로 늘렸을 때 자동 채점이 됩니다.

**해 보기:** 정상 사진으로도 돌려 보세요 — 아래 칸의 `ok_files[0]` 을 `ok_files[3]`, `def_files[5]` 처럼 바꿔 가며 몇 장을 봅니다.

In [ ]:
for f, 정답 in [(ok_files[0], "ok"), (def_files[1], "def"), (ok_files[1], "ok")]:
    r = chat_json(PROMPT, f)
    print(f"정답 {정답:>3} · 모델 {r.get('라벨')!s:>3} · {'맞음' if r.get('라벨') == 정답 else '틀림'} · {r.get('근거')}")

## [3-44] 상담 녹음 전사 (다시 하기)

녹음 `audio/call1_engine_start.wav` 는 **181초(약 3분)** 짜리 건설장비 고객센터 상담입니다.

화자 구분 전사 모델(`gpt-4o-transcribe-diarize`)은 **30초가 넘는 음성이면 `chunking_strategy` 를 꼭 지정**해야 합니다.
긴 녹음을 서버가 알아서 구간으로 나눠 받아쓰라는 뜻입니다. 어제 칸에는 이 한 줄이 없어 서버가 요청을 거절했습니다.
이 모델은 화자 이름을 모르니 `A`, `B` 로 돌려줍니다 — 먼저 말한 쪽(전화를 받은 쪽)을 **상담원**, 다른 쪽을 **고객** 으로 바꿔 적습니다.

> 이 모델 이름이 계정에서 막혀 있으면, 아래 칸이 **① 일반 전사 → ② 전사 글을 상담원/고객으로 나누기** 두 단계로 자동으로 바꿔 진행합니다.

In [ ]:
WAV, TXT = "audio/call1_engine_start.wav", "audio/call1_engine_start.txt"

def _multipart(fields, file_field, file_path, mime):
    bd = uuid.uuid4().hex
    out = b""
    for k, v in fields.items():
        out += f"--{bd}\r\nContent-Disposition: form-data; name=\"{k}\"\r\n\r\n{v}\r\n".encode()
    out += (f"--{bd}\r\nContent-Disposition: form-data; name=\"{file_field}\"; "
            f"filename=\"{os.path.basename(file_path)}\"\r\nContent-Type: {mime}\r\n\r\n").encode()
    out += open(file_path, "rb").read() + f"\r\n--{bd}--\r\n".encode()
    return out, f"multipart/form-data; boundary={bd}"

def stt(fields):
    payload, ctype = _multipart(fields, "file", WAV, "audio/wav")
    req = urllib.request.Request("https://api.openai.com/v1/audio/transcriptions", data=payload,
                                 headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": ctype})
    return json.loads(_send(req, timeout=600))

def diarized_to_lines(res):
    """화자 A·B 구간 → '상담원: …' / '고객: …' 줄. 같은 사람이 이어 말하면 한 줄로 붙입니다."""
    segs = res.get("segments") or []
    first = segs[0].get("speaker") if segs else None
    lines = []
    for s in segs:
        who = "상담원" if s.get("speaker") == first else "고객"
        text = (s.get("text") or "").strip()
        if not text:
            continue
        if lines and lines[-1][0] == who:
            lines[-1][1] += " " + text
        else:
            lines.append([who, text])
    return "\n".join(f"{w}: {t}" for w, t in lines)

if PROVIDER != "openai":
    raise SystemExit("이 칸은 OpenAI 키(sk-…)로 실행합니다. 🔑 에 OPENAI_API_KEY 를 등록해 주세요.")

방법 = ""
try:
    res = stt({"model": OPENAI_STT_DIARIZE,
               "response_format": "diarized_json",
               "chunking_strategy": "auto"})          # ← 어제 빠졌던 한 줄 (30초 넘는 음성에 필수)
    transcript = diarized_to_lines(res)
    방법 = "화자 구분 전사 한 번"
except RuntimeError as e:
    print("화자 구분 전사가 막혀 두 단계로 바꿉니다.\n  이유:", str(e).splitlines()[0])
    plain = stt({"model": OPENAI_STT_PLAIN, "response_format": "json"}).get("text", "")
    r = chat_json("다음은 건설장비 고객센터 상담 녹음을 받아쓴 글입니다. 내용은 고치지 말고, "
                  "말한 사람에 따라 '상담원'과 '고객'으로 나눠 순서대로 적으세요. "
                  '{"lines": [{"who": "상담원|고객", "text": "…"}]} JSON 으로만 답하세요.\n\n' + plain)
    transcript = "\n".join(f"{l['who']}: {l['text']}" for l in r.get("lines", []))
    방법 = "일반 전사 → 화자 나누기 두 단계"

print(f"[{방법}]\n")
print(transcript[:900], "\n…")

줄마다 `상담원:` 또는 `고객:` 이 붙어 나오면 성공입니다.

**생각해 볼 것:** 모델은 목소리만으로 A·B 를 가릅니다. 누가 상담원인지는 모릅니다.
「먼저 말한 쪽 = 상담원」 은 우리가 정한 **현장 규칙**입니다. 고객이 먼저 말을 거는 녹음(발신 상담)이라면 이 규칙을 바꿔야 합니다.
AI 가 해 주는 부분과 사람이 정해 줘야 하는 부분을 이렇게 나눠 보는 것이 이번 실습의 요점입니다.

## [3-45] 문자 오차율(CER) — 전사가 얼마나 맞았나

녹음에는 대본(`audio/call1_engine_start.txt`)이 있습니다. 대본이 곧 정답지입니다.
화자 표시·띄어쓰기·문장부호를 빼고 글자만 비교해, 몇 글자를 고쳐야 정답이 되는지(편집거리)를 셉니다.

In [ ]:
import re
def norm(t):
    t = re.sub(r"^(상담원|고객)\s*:", "", t, flags=re.M)
    return re.sub(r"[\s.,?!~'\"()\-·…]", "", t)
ref, hyp = norm(open(TXT, encoding="utf-8").read()), norm(transcript)
prev = list(range(len(hyp) + 1))
for i, rc in enumerate(ref, 1):
    cur = [i] + [0] * len(hyp)
    for j, hc in enumerate(hyp, 1):
        cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (rc != hc))
    prev = cur
print(f"정답 {len(ref)}자 · 편집거리 {prev[-1]} · CER = {prev[-1]/len(ref):.2%}")

# 화자가 맞게 갈렸는지도 봅니다 — 대본의 화자 순서와 전사의 화자 순서
who_ref = [l.split(":")[0] for l in open(TXT, encoding="utf-8").read().splitlines() if ":" in l]
who_hyp = [l.split(":")[0] for l in transcript.splitlines() if ":" in l]
print(f"대본 {len(who_ref)}줄 · 전사 {len(who_hyp)}줄 · 앞 6줄 화자 — 대본 {who_ref[:6]} / 전사 {who_hyp[:6]}")

CER 이 몇 % 인지, 화자 순서가 대본과 얼마나 같은지 패들릿 **Day03 실습결과** 에 캡처로 올려 주세요.

- CER 이 5% 안팎이면 사람이 훑어 고칠 수준, 20% 를 넘으면 녹음 품질(잡음·겹친 말)을 먼저 봐야 합니다.
- 화자 줄 수가 대본보다 적으면 같은 사람의 말이 한 줄로 붙은 것이고, 많으면 한 사람의 말이 쪼개진 것입니다.